# Rizzvision v3 — Dataset Curation

Add these as input datasets before running:
- `nitin2807/deepfashion2-full`
- `nitin2807/fashionpedia-full`
- `nitin2807/imaterialist-fashion-2020`
- `nitin2807/openimages-footwear`

In [ ]:
import os, json, csv, random, shutil
import concurrent.futures
from pathlib import Path
from collections import defaultdict
import pandas as pd
from tqdm.notebook import tqdm

BASE       = '/kaggle/input/datasets/nitin2807'
DF2_DIR    = f'{BASE}/deepfashion2-full'
FP_DIR     = f'{BASE}/fashionpedia-full'
IMAT_DIR   = f'{BASE}/imaterialist-fashion-2020'
ZAPPOS_DIR = '/kaggle/input/datasets/aryashah2k/large-shoe-dataset-ut-zappos50k'
OUT_DIR    = '/kaggle/working'

LABELS        = ['tops', 'bottoms', 'footwear', 'outerwear', 'dress']
MAX_PER_CLASS = 25000
VAL_FRAC      = 0.10
TEST_FRAC     = 0.10
SEED          = 42
NUM_WORKERS   = 8

os.makedirs(f'{OUT_DIR}/images/train', exist_ok=True)
os.makedirs(f'{OUT_DIR}/images/val',   exist_ok=True)
os.makedirs(f'{OUT_DIR}/images/test',  exist_ok=True)

for name, d in [('DF2', DF2_DIR), ('FP', FP_DIR), ('iMat', IMAT_DIR), ('Zappos', ZAPPOS_DIR)]:
    items = os.listdir(d) if os.path.exists(d) else []
    print(f'{name}: {items[:4]}')
print('Config ready')

In [ ]:
DF2_CAT_TO_LABEL = {
    1:'tops', 2:'tops', 5:'tops', 6:'tops',
    3:'outerwear', 4:'outerwear',
    7:'bottoms', 8:'bottoms', 9:'bottoms',
    10:'dress', 11:'dress', 12:'dress', 13:'dress',
}

# Shared supercategory map — covers both Fashionpedia and iMaterialist casing variants
SUPER_TO_LABEL = {
    'upperbody':'tops',      'Upper-body':'tops',     'Upperbody':'tops',
    'lowerbody':'bottoms',   'Lower-body':'bottoms',  'Lowerbody':'bottoms',
    'wholebody':'dress',     'Whole-body':'dress',    'Wholebody':'dress',
    'outerwear':'outerwear', 'Outerwear':'outerwear',
    'feet':'footwear',       'Feet':'footwear',
    'footwear':'footwear',   'Footwear':'footwear',   # Fashionpedia variants
    'shoes':'footwear',      'Shoes':'footwear',
}
OI_FOOTWEAR = {'/m/0fly7','/m/01b638','/m/03yfpv','/m/02gzp','/m/0_cp5'}

def make_row(filepath, label_set):
    row = {'filepath': str(filepath)}
    for l in LABELS:
        row[l] = 1 if l in label_set else 0
    return row

# Per-source footwear counter so we can see exactly where footwear comes from
footwear_sources = {}

print('Mappings ready')
print(f'SUPER_TO_LABEL keys: {list(SUPER_TO_LABEL.keys())}')

In [ ]:
# ── DeepFashion2 (parallel) ────────────────────────────────────────────────
records = []

def process_df2_anno(af, image_dir):
    try: data = json.loads(af.read_text())
    except: return None
    if not isinstance(data, dict): return None  # skip malformed files
    label_set = set()
    for k, v in data.items():
        if k.startswith('item') and isinstance(v, dict):
            lbl = DF2_CAT_TO_LABEL.get(v.get('category_id'))
            if lbl: label_set.add(lbl)
    if not label_set: return None
    src = next((image_dir / f'{af.stem}{ext}' for ext in ('.jpg','.JPG','.png')
                if (image_dir / f'{af.stem}{ext}').exists()), None)
    return make_row(src, label_set) if src else None

for split, anno_subdir, img_subdir in [
    ('train',      'train',               'train'),
    ('validation', 'json_for_validation', 'validation'),
]:
    base = Path(DF2_DIR) / anno_subdir
    anno_dir  = base / 'annos' if (base / 'annos').exists() else base
    image_dir = Path(DF2_DIR) / img_subdir / 'image'
    if not image_dir.exists(): image_dir = Path(DF2_DIR) / img_subdir
    anno_files = sorted(anno_dir.glob('*.json')) if anno_dir.exists() else []
    if not anno_files:
        print(f'[DF2] {split}: no annotation files found at {anno_dir}'); continue
    print(f'[DF2] {split}: {len(anno_files):,} files, images at {image_dir}')
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = {ex.submit(process_df2_anno, af, image_dir): af for af in anno_files}
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc=f'DF2/{split}'):
            result = fut.result()
            if result: records.append(result)

print(f'After DF2: {len(records):,} records')

In [ ]:
# ── Fashionpedia (parallel) ────────────────────────────────────────────────
fp_anno_dir = Path(FP_DIR) / 'annotations'
fp_img_root = Path(FP_DIR) / 'images'
fp_img_dirs = [fp_img_root] + ([p for p in fp_img_root.iterdir() if p.is_dir()] if fp_img_root.exists() else [])

def process_fp_item(img_id, label_set, id_to_file, img_dirs):
    fname = id_to_file.get(img_id)
    if not fname: return None
    src = next((d / fname for d in img_dirs if (d / fname).exists()), None)
    return make_row(src, label_set) if src else None

fp_footwear = 0
for anno_file in ('instances_attributes_train2020.json', 'instances_attributes_val2020.json'):
    anno_path = fp_anno_dir / anno_file
    if not anno_path.exists():
        print(f'[FP] {anno_file} not found, skipping'); continue
    data = json.loads(anno_path.read_text())

    # Print actual supercategories present so we can verify mapping coverage
    all_supers = sorted(set(cat.get('supercategory','') for cat in data.get('categories', [])))
    print(f'[FP] {anno_file} supercategories: {all_supers}')

    cat_map = {cat['id']: SUPER_TO_LABEL[cat['supercategory']]
               for cat in data.get('categories', [])
               if cat.get('supercategory') in SUPER_TO_LABEL}
    img_labels = defaultdict(set)
    for ann in data.get('annotations', []):
        lbl = cat_map.get(ann.get('category_id'))
        if lbl: img_labels[ann['image_id']].add(lbl)
    id_to_file = {img['id']: img['file_name'] for img in data.get('images', [])}

    fw_count = sum(1 for ls in img_labels.values() if 'footwear' in ls)
    print(f'[FP] {anno_file}: {len(img_labels):,} labelled images  |  footwear images: {fw_count:,}')
    fp_footwear += fw_count

    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = [ex.submit(process_fp_item, img_id, label_set, id_to_file, fp_img_dirs)
                for img_id, label_set in img_labels.items()]
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc='FP'):
            result = fut.result()
            if result: records.append(result)

footwear_sources['fashionpedia'] = fp_footwear
print(f'After Fashionpedia: {len(records):,} records  |  FP footwear: {fp_footwear:,}')

In [ ]:
# ── iMaterialist 2020 (parallel) ───────────────────────────────────────────
csv.field_size_limit(10 * 1024 * 1024)

label_desc_path = Path(IMAT_DIR) / 'label_descriptions.json'
train_csv_path  = Path(IMAT_DIR) / 'train.csv'
train_imgs_path = Path(IMAT_DIR) / 'train'

desc = json.loads(label_desc_path.read_text())

# Print actual supercategories present in iMat
all_supers = sorted(set(cat.get('supercategory','') for cat in desc.get('categories', [])))
print(f'[iMat] supercategories: {all_supers}')

cat_map = {int(cat['id']): SUPER_TO_LABEL[cat['supercategory']]
           for cat in desc.get('categories', [])
           if cat.get('supercategory') in SUPER_TO_LABEL}
print(f'[iMat] {len(cat_map)} mapped label IDs')

img_labels = defaultdict(set)
with open(train_csv_path) as f:
    for row in csv.DictReader(f):
        for cid_str in row.get('ClassId','').replace(',',' ').split():
            try:
                lbl = cat_map.get(int(cid_str.split('_')[0]))
                if lbl: img_labels[row['ImageId']].add(lbl)
            except ValueError: pass

imat_footwear = sum(1 for ls in img_labels.values() if 'footwear' in ls)
print(f'[iMat] {len(img_labels):,} labelled images  |  footwear images: {imat_footwear:,}')

def process_imat_item(img_id, label_set, imgs_path):
    src = next((imgs_path / f'{img_id}{ext}' for ext in ('.jpg','.png')
                if (imgs_path / f'{img_id}{ext}').exists()), None)
    return make_row(src, label_set) if src else None

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
    futs = [ex.submit(process_imat_item, img_id, label_set, train_imgs_path)
            for img_id, label_set in img_labels.items()]
    for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc='iMat'):
        result = fut.result()
        if result: records.append(result)

footwear_sources['imat'] = imat_footwear
print(f'After iMat: {len(records):,} records  |  iMat footwear: {imat_footwear:,}')

In [ ]:
# ── Footwear: UT Zappos50K (catalog — studio product photos) ───────────────
ZAPPOS_DIR = '/kaggle/input/datasets/aryashah2k/large-shoe-dataset-ut-zappos50k'

if not Path(ZAPPOS_DIR).exists():
    print('[Zappos] dataset not found — add aryashah2k/large-shoe-dataset-ut-zappos50k as input')
    footwear_sources['zappos'] = 0
else:
    imgs = list(Path(ZAPPOS_DIR).rglob('*.jpg')) + list(Path(ZAPPOS_DIR).rglob('*.png'))
    print(f'[Zappos] found {len(imgs):,} footwear images (catalog/studio)')
    imgs = imgs[:15000]  # cap at 15K

    def add_zappos(src):
        return make_row(src, {'footwear'}) if src.exists() else None

    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = [ex.submit(add_zappos, src) for src in imgs]
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc='Zappos'):
            result = fut.result()
            if result: records.append(result)
    footwear_sources['zappos'] = len(imgs)

print(f'\nFootwear source breakdown:')
for src, count in footwear_sources.items():
    print(f'  {src:<15}: {count:,}')
total_fw = sum(footwear_sources.values())
in_wild  = total_fw - footwear_sources.get('zappos', 0)
print(f'  {"TOTAL":<15}: {total_fw:,}  (in-the-wild: {in_wild:,}  |  catalog: {footwear_sources.get("zappos",0):,})')
print(f'\nAfter all footwear: {len(records):,} records')

In [ ]:
# ── Balance ────────────────────────────────────────────────────────────────
rng = random.Random(SEED)
rng.shuffle(records)
counts = defaultdict(int)
kept = []
for rec in records:
    present = [l for l in LABELS if rec[l] == 1]
    if not present: continue
    if any(counts[l] < MAX_PER_CLASS for l in present):
        kept.append(rec)
        for l in present: counts[l] += 1

print(f'Total after balancing: {len(kept):,}')
for l in LABELS:
    print(f'  {l:<12}: {counts[l]:,}')

In [ ]:
# ── Split + copy images (parallel) ────────────────────────────────────────
rng2 = random.Random(SEED)
rng2.shuffle(kept)
n = len(kept)
n_test = int(n * TEST_FRAC)
n_val  = int(n * VAL_FRAC)
splits_data = {
    'test':  kept[:n_test],
    'val':   kept[n_test:n_test+n_val],
    'train': kept[n_test+n_val:],
}
print('Split sizes:', {k: len(v) for k,v in splits_data.items()})

def copy_image(rec, dst_dir):
    src = Path(rec['filepath'])
    dst = dst_dir / src.name
    if not dst.exists(): shutil.copy2(src, dst)
    rec['filepath'] = str(dst)

for split_name, split_records in splits_data.items():
    dst_dir = Path(OUT_DIR) / 'images' / split_name
    dst_dir.mkdir(parents=True, exist_ok=True)
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futs = [ex.submit(copy_image, rec, dst_dir) for rec in split_records]
        for fut in tqdm(concurrent.futures.as_completed(futs), total=len(futs), desc=f'Copying {split_name}'):
            fut.result()

print('Images copied')

In [ ]:
# ── Write CSVs + stats ─────────────────────────────────────────────────────
for split_name, split_records in splits_data.items():
    out_path = f'{OUT_DIR}/labels_{split_name}.csv'
    pd.DataFrame(split_records).to_csv(out_path, index=False)
    print(f'Wrote {len(split_records):,} rows → {out_path}')

stats = {
    'total': len(kept),
    'splits': {k: len(v) for k,v in splits_data.items()},
    'label_counts': {
        sn: {l: sum(r[l] for r in recs) for l in LABELS}
        for sn, recs in splits_data.items()
    }
}
with open(f'{OUT_DIR}/stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('\nFinal stats:')
print(json.dumps(stats, indent=2))
print('\nDone! Save output as dataset: rizzvision-clothing-dataset-v3')